In [ ]:
import pandas as pd
import os
import json
from importlib import reload
curr_wd = os.path.abspath(os.getcwd())
print(curr_wd)



In [ ]:
###### MULTIPLE FILE IMPORT
import src.blast_parser as blast_parser
# Reload the module after making changes
reload(blast_parser)

folder = "/mnt/d/phd/scripts/16_ev_signature_predictor/data/blastp"

df_pos_raw_json = blast_parser.load_blast_group(folder, "fullseq_pos_alignment_primata_1.json", "fullseq_pos_alignment_primata_2.json", "fullseq_pos_alignment_primata_3.json")
df_neg_raw_json = blast_parser.load_blast_group(folder, "fullseq_neg_alignment_primata_1.json",  "fullseq_neg_alignment_primata_2.json")


In [ ]:
print(df_pos_raw_json.head())
print(df_pos_raw_json.shape)

In [ ]:
def expand_hits_to_motifs(df_blast, df_windows):
    """
    Join protein-level BLAST hits to motif-level metadata.
    Drops motif coordinates from blast header in favour of 
    the accurate per-motif coordinates from df_windows.
    """
    df_blast = df_blast.copy()
    df_blast["UniqueID"] = df_blast["query_title"].str.split("_").str[0]

    # Drop the protein-level motif coords from the BLAST header — 
    # they are meaningless for multi-motif proteins
    df_blast = df_blast.drop(columns=["motif_start", "motif_end"])

    df_expanded = df_windows.merge(
        df_blast,
        on="UniqueID",
        how="left"
    )

    return df_expanded


# 1. Expand first — motif_start/motif_end now come from df_windows
df_windows_pos = pd.read_pickle("/mnt/d/phd/scripts/15_vertical_evolutionary_analysis_orthologs/data/output/RG_motifs_full_sequence_win5_pos_metadata.pkl")
df_windows_neg = pd.read_pickle("/mnt/d/phd/scripts/15_vertical_evolutionary_analysis_orthologs/data/output/RG_motifs_full_sequence_win5_neg_metadata.pkl")

df_pos_expanded = expand_hits_to_motifs(df_pos_raw_json, df_windows_pos)
df_neg_expanded = expand_hits_to_motifs(df_neg_raw_json, df_windows_neg)

import pandas as pd
import numpy as np


# ---------------------------------------------------------------------------
# helpers
# ---------------------------------------------------------------------------
import pandas as pd
import numpy as np


def compute_rg_window_columns(df: pd.DataFrame) -> pd.DataFrame:
    """
    Adds two columns to the homolog dataframe:

    rg_window_coverage : str
        'full'    - BLAST alignment spans the entire RG window
        'partial' - BLAST alignment overlaps the RG window but doesn't cover it fully
        'none'    - BLAST alignment does not overlap the RG window at all

    rg_window_gap_fraction_bin : str
        Quartile bin of gap fraction in hseq over the covered RG window region.
        '0-25%', '25-50%', '50-75%', '75-100%'
        NaN for rows where rg_window_coverage == 'none'

    Coordinate assumptions:
        win_start_x, win_end_x  : 0-based, end-exclusive (Python slice)
        query_from              : 1-based, inclusive  -> converted to 0-based
        query_to                : 1-based, inclusive  -> converted to 0-based exclusive
        qseq / hseq             : aligned strings starting at query_from (may contain '-')
    """
    df = df.copy()

    results_coverage = []
    results_gap_bin  = []

    for _, row in df.iterrows():
        win_start = row["win_start_x"]  # 0-based, exclusive end
        win_end   = row["win_end_x"]

        # convert BLAST coords to 0-based exclusive
        aln_start = int(row["query_from"]) - 1
        aln_end   = int(row["query_to"])    # already exclusive after this conversion

        # --- coverage category ---
        overlap_start = max(win_start, aln_start)
        overlap_end   = min(win_end,   aln_end)
        overlap_len   = overlap_end - overlap_start  # negative if no overlap

        if overlap_len <= 0:
            results_coverage.append("none")
            results_gap_bin.append(np.nan)
            continue

        win_len = win_end - win_start
        if overlap_start <= win_start and overlap_end >= win_end:
            coverage = "full"
        else:
            coverage = "partial"

        results_coverage.append(coverage)

        # --- gap fraction in hseq over the overlapping window ---
        # qseq/hseq are alignment strings that may contain '-'
        # they start at aln_start on the protein coordinate axis
        # we need to walk qseq to find the alignment columns that correspond
        # to protein positions [overlap_start, overlap_end)

        qseq = row["qseq"]
        hseq = row["hseq"]

        # walk along qseq, tracking protein position
        protein_pos = aln_start
        hseq_window_chars = []

        for aln_col, (q_char, h_char) in enumerate(zip(qseq, hseq)):
            if protein_pos >= overlap_end:
                break
            if q_char != "-":
                # this alignment column corresponds to a real protein position
                if protein_pos >= overlap_start:
                    hseq_window_chars.append(h_char)
                protein_pos += 1
            # if q_char is '-' (insertion in subject), protein_pos doesn't advance
            # we don't count these columns since they have no query coordinate

        if len(hseq_window_chars) == 0:
            results_gap_bin.append(np.nan)
            continue

        gap_fraction = sum(1 for c in hseq_window_chars if c == "-") / len(hseq_window_chars)

        if gap_fraction <= 0.25:
            bin_label = "0-25%"
        elif gap_fraction <= 0.50:
            bin_label = "25-50%"
        elif gap_fraction <= 0.75:
            bin_label = "50-75%"
        else:
            bin_label = "75-100%"

        results_gap_bin.append(bin_label)

    df["rg_window_coverage"]       = results_coverage
    df["rg_window_gap_fraction_bin"] = results_gap_bin

    # make gap bin an ordered categorical for easy filtering/sorting
    gap_bin_order = ["0-25%", "25-50%", "50-75%", "75-100%"]
    df["rg_window_gap_fraction_bin"] = pd.Categorical(
        df["rg_window_gap_fraction_bin"],
        categories=gap_bin_order,
        ordered=True
    )

    return df


def summarise_rg_window_columns(df: pd.DataFrame) -> None:
    """Print a quick summary after adding the two columns."""
    print("=== rg_window_coverage ===")
    print(df["rg_window_coverage"].value_counts())
    print()
    print("=== rg_window_gap_fraction_bin ===")
    print(df["rg_window_gap_fraction_bin"].value_counts().sort_index())
    print()
    print("=== coverage x gap_bin crosstab ===")
    print(pd.crosstab(df["rg_window_coverage"], df["rg_window_gap_fraction_bin"].astype(str)))

def _query_centric(df: pd.DataFrame) -> pd.DataFrame:
    """
    For each UniqueID, compute:
      - n_homologs         : total homolog rows
      - n_unique_accessions: accessions that appear in NO other UniqueID
      - pct_unique         : n_unique_accessions / n_homologs * 100
    """
    # for each accession, how many distinct UniqueIDs carry it?
    acc_uid_counts = (
        df.groupby("hit_accession")["UniqueID"]
        .nunique()
        .rename("n_queries_with_acc")
    )
    df = df.join(acc_uid_counts, on="hit_accession")

    per_query = (
        df.groupby("UniqueID")
        .apply(lambda g: pd.Series({
            "n_homologs":          len(g),
            "n_unique_accessions": (g["n_queries_with_acc"] == 1).sum(),
        }))
        .reset_index()
    )
    per_query["pct_unique"] = (
        per_query["n_unique_accessions"] / per_query["n_homologs"] * 100
    ).round(2)
    return per_query


def _homolog_centric(df: pd.DataFrame) -> pd.Series:
    """
    For each hit_accession, count how many distinct UniqueIDs it appears in.
    Returns a Series of those counts (one value per accession).
    """
    return (
        df.groupby("hit_accession")["UniqueID"]
        .nunique()
        .rename("n_queries")
    )


def _summarise_query_centric(qc: pd.DataFrame, label: str) -> pd.Series:
    return pd.Series({
        "filter_step":           label,
        "n_queries":             len(qc),
        "total_homolog_rows":    qc["n_homologs"].sum(),
        "median_pct_unique":     round(qc["pct_unique"].median(), 2),
        "mean_pct_unique":       round(qc["pct_unique"].mean(), 2),
        "pct_queries_all_unique": (qc["pct_unique"] == 100).mean() * 100,
    })


def _summarise_homolog_centric(hc: pd.Series, label: str) -> pd.Series:
    return pd.Series({
        "filter_step":               label,
        "n_unique_accessions":       len(hc),
        "pct_exclusive_to_1_query":  (hc == 1).mean() * 100,
        "pct_in_2_to_5_queries":     ((hc >= 2) & (hc <= 5)).mean() * 100,
        "pct_in_6_to_20_queries":    ((hc >= 6) & (hc <= 20)).mean() * 100,
        "pct_in_gt20_queries":       (hc > 20).mean() * 100,
        "median_queries_per_acc":    hc.median(),
        "max_queries_per_acc":       hc.max(),
    })


# ---------------------------------------------------------------------------
# main
# ---------------------------------------------------------------------------

def compute_filtering_funnel(df: pd.DataFrame) -> dict:
    """
    Computes query-centric and homolog-centric redundancy summaries across
    a filtering funnel of increasing stringency.

    Filter steps:
        1. all            : no filtering
        2. full           : rg_window_coverage == 'full'
        3. full + 0-25%   : full coverage + gap bin <= '0-25%'  (cumulative)
        4. full + 0-50%   : full coverage + gap bin <= '25-50%' (cumulative)
        5. full + 0-75%   : full coverage + gap bin <= '50-75%' (cumulative)

    Returns
    -------
    dict with keys:
        'query_centric_summary'   : pd.DataFrame  (one row per filter step)
        'homolog_centric_summary' : pd.DataFrame  (one row per filter step)
        'query_centric_detail'    : dict of pd.DataFrames (per-query detail per step)
        'homolog_centric_detail'  : dict of pd.Series    (per-accession counts per step)
    """
    gap_order = ["0-25%", "25-50%", "50-75%", "75-100%"]

    filter_steps = {
        "1_all":          df,
        "2_full":         df[df["rg_window_coverage"] == "full"],
        "3_full_gap0-25": df[
            (df["rg_window_coverage"] == "full") &
            (df["rg_window_gap_fraction_bin"] == "0-25%")
        ],
        "4_full_gap0-50": df[
            (df["rg_window_coverage"] == "full") &
            (df["rg_window_gap_fraction_bin"].isin(["0-25%", "25-50%"]))
        ],
        "5_full_gap0-75": df[
            (df["rg_window_coverage"] == "full") &
            (df["rg_window_gap_fraction_bin"].isin(["0-25%", "25-50%", "50-75%"]))
        ],
    }

    qc_summaries  = []
    hc_summaries  = []
    qc_details    = {}
    hc_details    = {}

    for label, subset in filter_steps.items():
        qc = _query_centric(subset)
        hc = _homolog_centric(subset)

        qc_summaries.append(_summarise_query_centric(qc, label))
        hc_summaries.append(_summarise_homolog_centric(hc, label))
        qc_details[label] = qc
        hc_details[label] = hc

    return {
        "query_centric_summary":   pd.DataFrame(qc_summaries).set_index("filter_step"),
        "homolog_centric_summary": pd.DataFrame(hc_summaries).set_index("filter_step"),
        "query_centric_detail":    qc_details,
        "homolog_centric_detail":  hc_details,
    }


def print_funnel_report(results: dict) -> None:
    """Pretty-print the funnel summary tables."""
    pd.set_option("display.float_format", "{:.2f}".format)
    pd.set_option("display.max_columns", 20)
    pd.set_option("display.width", 120)

    print("=" * 80)
    print("QUERY-CENTRIC SUMMARY")
    print("  median/mean % of homologs that are exclusive to that query")
    print("  pct_queries_all_unique: % of UniqueIDs where ALL homologs are exclusive")
    print("=" * 80)
    print(results["query_centric_summary"].T)

    print()
    print("=" * 80)
    print("HOMOLOG-CENTRIC SUMMARY")
    print("  distribution of how many queries each accession appears in")
    print("=" * 80)
    print(results["homolog_centric_summary"].T)


def get_per_query_detail(results: dict, step: str) -> pd.DataFrame:
    """
    Return the per-UniqueID detail table for a given filter step.
    step: one of '1_all', '2_full', '3_full_gap0-25', '4_full_gap0-50', '5_full_gap0-75'
    """
    return results["query_centric_detail"][step]


def get_shared_accessions(results: dict, step: str, min_queries: int = 2) -> pd.DataFrame:
    """
    Return accessions shared across >= min_queries UniqueIDs for a given step,
    sorted by sharing count descending.
    """
    hc = results["homolog_centric_detail"][step]
    shared = hc[hc >= min_queries].sort_values(ascending=False).reset_index()
    shared.columns = ["hit_accession", "n_queries"]
    return shared

import pandas as pd
import numpy as np


def assign_shared_accessions(df: pd.DataFrame, random_seed: int = 42) -> pd.DataFrame:
    """
    For each hit_accession that appears under more than one UniqueID, assign it
    to the UniqueID with the highest mean bit_score for that accession. If two
    UniqueIDs tie on bit_score, one is chosen at random.

    Accessions appearing under multiple orig_motif_index values of the SAME
    UniqueID are always kept (not considered redundant).

    Parameters
    ----------
    df          : dataframe after compute_rg_window_columns(), filtered to
                  whatever coverage/gap step you want to apply this to
    random_seed : for reproducibility of random tie-breaking

    Returns
    -------
    df_out      : filtered dataframe where each hit_accession appears under
                  at most one UniqueID (but may appear under multiple motifs
                  within that UniqueID)
    report      : dict with summary statistics
    """
    rng = np.random.default_rng(random_seed)

    # --- step 1: identify accessions shared across multiple UniqueIDs ---
    acc_uid = (
        df.groupby("hit_accession")["UniqueID"]
        .nunique()
        .rename("n_unique_ids")
    )
    shared_accessions = acc_uid[acc_uid > 1].index

    n_shared = len(shared_accessions)
    n_total_acc = acc_uid.shape[0]

    if n_shared == 0:
        report = {
            "n_total_accessions":   n_total_acc,
            "n_shared_accessions":  0,
            "n_randomly_assigned":  0,
            "pct_randomly_assigned": 0.0,
        }
        return df.copy(), report

    # --- step 2: for shared accessions, compute mean bit_score per (accession, UniqueID) ---
    shared_df = df[df["hit_accession"].isin(shared_accessions)].copy()

    mean_bitscore = (
        shared_df.groupby(["hit_accession", "UniqueID"])["bit_score"]
        .mean()
        .reset_index()
        .rename(columns={"bit_score": "mean_bit_score"})
    )

    # --- step 3: for each accession, find the best UniqueID ---
    assigned_uid   = {}
    n_random       = 0

    for acc, grp in mean_bitscore.groupby("hit_accession"):
        max_score = grp["mean_bit_score"].max()
        candidates = grp[grp["mean_bit_score"] == max_score]["UniqueID"].tolist()

        if len(candidates) == 1:
            assigned_uid[acc] = candidates[0]
        else:
            # tie — assign randomly
            assigned_uid[acc] = rng.choice(candidates)
            n_random += 1

    pct_random = round(n_random / n_shared * 100, 2) if n_shared > 0 else 0.0

    # --- step 4: build the filtered dataframe ---
    # rows where accession is NOT shared: always keep
    df_unshared = df[~df["hit_accession"].isin(shared_accessions)].copy()

    # rows where accession IS shared: keep only if UniqueID matches assignment
    df_shared = df[df["hit_accession"].isin(shared_accessions)].copy()
    df_shared["_assigned_uid"] = df_shared["hit_accession"].map(assigned_uid)
    df_shared = df_shared[df_shared["UniqueID"] == df_shared["_assigned_uid"]]
    df_shared = df_shared.drop(columns=["_assigned_uid"])

    df_out = pd.concat([df_unshared, df_shared], ignore_index=True)

    # --- step 5: compile report ---
    report = {
        "n_total_accessions":    n_total_acc,
        "n_shared_accessions":   n_shared,
        "n_randomly_assigned":   n_random,
        "pct_randomly_assigned": pct_random,
        "n_rows_before":         len(df),
        "n_rows_after":          len(df_out),
        "n_rows_removed":        len(df) - len(df_out),
    }

    return df_out, report


def print_assignment_report(report: dict) -> None:
    print("=" * 60)
    print("ACCESSION ASSIGNMENT REPORT")
    print("=" * 60)
    print(f"  Total unique accessions      : {report['n_total_accessions']}")
    print(f"  Shared across >1 UniqueID    : {report['n_shared_accessions']}")
    print(f"  Randomly assigned (ties)     : {report['n_randomly_assigned']}"
          f"  ({report['pct_randomly_assigned']}%)")
    print(f"  Rows before assignment       : {report['n_rows_before']}")
    print(f"  Rows after assignment        : {report['n_rows_after']}")
    print(f"  Rows removed                 : {report['n_rows_removed']}")
    print("=" * 60)



# from rg_window_filter import compute_rg_window_columns, summarise_rg_window_columns
df_pos_expanded_categorized = compute_rg_window_columns(df_pos_expanded)
df_neg_expanded_categorized = compute_rg_window_columns(df_neg_expanded)

df_clean_pos = (
    df_pos_expanded_categorized.sort_values("bit_score", ascending=False)
    .drop_duplicates(subset=["hit_accession", "UniqueID", "orig_motif_index"])
    .reset_index(drop=True)
)
df_clean_pos

df_clean_neg = (
    df_neg_expanded_categorized.sort_values("bit_score", ascending=False)
    .drop_duplicates(subset=["hit_accession", "UniqueID", "orig_motif_index"])
    .reset_index(drop=True)
)
df_clean_neg

# example: apply to full coverage + 0-25% gap bin
df_full_pos = df_clean_pos[
    (df_clean_pos["rg_window_coverage"] == "full")].copy()

df_full_pos_final, report = assign_shared_accessions(df_full_pos)
print_assignment_report(report)

# ============================================================
# ACCESSION ASSIGNMENT REPORT
# ============================================================
#   Total unique accessions      : 18930
#   Shared across >1 UniqueID    : 2931
#   Randomly assigned (ties)     : 80  (2.73%)
#   Rows before assignment       : 49036
#   Rows after assignment        : 37046
#   Rows removed                 : 11990
# ============================================================

# example: apply to full coverage + 0-25% gap bin
df_full_neg = df_clean_neg[
    (df_clean_neg["rg_window_coverage"] == "full")].copy()

df_full_neg_final, report = assign_shared_accessions(df_full_neg)
print_assignment_report(report)


In [ ]:
# Import the functions
from src.clean_up_blast_tools.expand import expand_hits_to_motifs
from src.clean_up_blast_tools.coverage import compute_rg_window_columns
from src.clean_up_blast_tools.assignment import assign_shared_accessions


df_windows_pos = pd.read_pickle("/mnt/d/phd/scripts/15_vertical_evolutionary_analysis_orthologs/data/output/RG_motifs_full_sequence_win5_pos_metadata.pkl")
df_windows_neg = pd.read_pickle("/mnt/d/phd/scripts/15_vertical_evolutionary_analysis_orthologs/data/output/RG_motifs_full_sequence_win5_neg_metadata.pkl")

df_pos_expanded = expand_hits_to_motifs(df_pos_raw_json, df_windows_pos)
df_neg_expanded = expand_hits_to_motifs(df_neg_raw_json, df_windows_neg)

